# DFT расчёт HEA L12 (32 атома) — CPU Версия

**Инструкция:**
1. Загрузи `HEA_L12_SQS_32atoms.cif` и папку `pseudo`.
2. Запусти ячейки по очереди.

Этот ноутбук использует стандартный пакет QE из репозитория Ubuntu. Он **стабилен** и не требует компиляции.

In [ ]:
# 1. Установка QE (CPU версия)
!apt-get update -qq
!apt-get install -y quantum-espresso > /dev/null
!pip install -q pymatgen
print("✅ QE установлен.")

In [ ]:
# 2. Генерация input и запуск
from pymatgen.core import Structure
import re, os

cif_file = "HEA_L12_SQS_32atoms.cif"
pseudo_map = {
    "Al": "Al.pbesol-n-kjpaw_psl.1.0.0.UPF",
    "Ti": "Ti.upf",
    "Cr": "Cr.upf",
    "Fe": "Fe.pbesol-spn-kjpaw_psl.0.2.1.UPF",
    "Co": "Co.upf",
    "Ni": "Ni.upf"
}
masses = {"Al": 26.98, "Ti": 47.87, "Cr": 52.00, "Fe": 55.85, "Co": 58.93, "Ni": 58.69}

struct = Structure.from_file(cif_file)

inp = f"""&CONTROL
    calculation = 'vc-relax'
    prefix = 'HEA_L12_32'
    pseudo_dir = './pseudo'
    outdir = './tmp'
    etot_conv_thr = 1.0d-4
    forc_conv_thr = 1.0d-3
    nstep = 200
    verbosity = 'high'
/
&SYSTEM
    ibrav = 0
    nat = {struct.num_sites}
    ntyp = {len(pseudo_map)}
    ecutwfc = 60
    ecutrho = 480
    occupations = 'smearing'
    smearing = 'methfessel-paxton'
    degauss = 0.02
    input_dft = 'pbesol'
/
&ELECTRONS
    conv_thr = 1.0d-8
    mixing_beta = 0.3
    electron_maxstep = 200
/
&IONS
    ion_dynamics = 'bfgs'
/
&CELL
    cell_dynamics = 'bfgs'
    press = 0.0
/
ATOMIC_SPECIES
"""

for el, fname in pseudo_map.items():
    inp += f" {el}  {masses[el]}  {fname}\n"

inp += "CELL_PARAMETERS angstrom\n"
for v in struct.lattice.matrix:
    inp += f" {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n"

inp += "ATOMIC_POSITIONS crystal\n"
for site in struct:
    sym = site.species.elements[0].symbol
    fc = site.frac_coords
    inp += f" {sym}  {fc[0]:.6f} {fc[1]:.6f} {fc[2]:.6f}\n"

inp += "K_POINTS automatic\n 4 4 4  0 0 0\n"

with open("vc-relax.in", "w") as f:
    f.write(inp)

print(f"Создан vc-relax.in ({struct.num_sites} атомов). Запуск...")

# Запуск с выводом в реальном времени
!pw.x < vc-relax.in 2>&1 | tee vc-relax.out

# Результат
with open("vc-relax.out", "r") as f:
    out = f.read()
energy = re.search(r'!\s+total energy\s+=\s+([\-\d\.]+)\s+Ry', out)
if energy:
    print(f"\n✅ Энергия: {energy.group(1)} Ry")
else:
    print("\n❌ Энергия не найдена.")